In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)

In [ ]:
# df_companies.reset_index(inplace=True)#.loc[:"2020-06-01"].tail(50)#.groupby("date").sum().plot()
# # df_companies.reset_index().loc[:"2020-06-01"].groupby("symbol").mean().reset_index().plot(x="marketCap", y="symbol", kind="scatter",figsize=(10,30))#.groupby("symbol").plot()
# df_companies.groupby("date").sum().loc[2]#loc[df_companies.index.get_level_values("date")==date].values.sum()

In [ ]:
mean_prc = df_articles['PressReleaseCount'].mean()
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
df_companies=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SPXCompaniesByMarketCap.pkl")

In [ ]:
def marketCapFactor(companies,date):
    marketcap = 0
    for company in companies:
        try:
            marketcap += df_companies.loc[company, date].values[0]
        except:
            print("Exception")
            print(company)
            print(date)
    totalMarketCap = df_companies.loc[df_companies.index.get_level_values("date")==date].values.sum()
    print(totalMarketCap)
    return (marketcap/totalMarketCap)

# 1) DATE

In [ ]:
import pysentiment2 as ps
hiv4 = ps.HIV4()
df  = pd.DataFrame()

df[['Negative']] = ''
df[['Positive']] = ''
df[['Sentiment']] = ''
df[['Adjectives']] = ''
df[['Verbs']] = ''
df[['Adverbs']] = ''
df[['Nouns']] = ''
df[["Articles"]] = ''
df[["Polarity"]] = ''
df[["Subjectivity"]] = ''
for date, row in df_articles.T.iteritems():
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    prc = row['PressReleaseCount']
    sentence = str(preprocessing_text(sentence))
    print("#0")
    sentence = sentence.lower()
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date,"Articles"] = sentence
    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)

    df.at[date, 'Negative'] = score["Negative"]
    df.at[date, 'Positive'] = score["Positive"]
    df.at[date, 'Polarity'] = score["Polarity"]
    df.at[date, 'Subjectivity'] = score["Subjectivity"]
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = marketCapFactor(companieslist, date)
    print("#2")
    df.at[date, 'MarketCapFactor'] = mcf
    if ((score['Positive'] + score['Negative']) != 0):
        number = 100*(score['Positive'] / (score['Positive'] + score['Negative']))
        if (number != ''):
            df.at[date, 'Sentiment'] = (((100*(score['Negative'] / (score['Positive'] + score['Negative'])))*(prc/mean_prc)))*mcf

df['Sentiment'].apply(np.float)
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
df['Sentiment'] = df['Sentiment'].apply(np.float)


In [ ]:
df['Sentiment'].apply(np.float)
# df_articles.drop(columns={"Articles"},inplace=True)
# df = df.join(df_articles)
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentGeneralInquirer.pkl")

# 2) HOUR

In [ ]:
del df 

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDateHourDF.pkl', compression=None)

In [ ]:
import pysentiment2 as ps
hiv4 = ps.HIV4()
rows_list = []
for date, row in df_articles.T.iteritems():
    Hour = int(row["Hour"])
    print(Hour)
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")

    prc = row['PressReleaseCount']
    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)
    Negative = score['Negative']
    Positive = score['Positive']
    Polarity = score['Polarity']
    Subjectivity = score['Subjectivity']
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = marketCapFactor(companieslist, date)
    MarketCapFactor = mcf
    Sentiment=0
    if ((Positive + Negative) != 0):
        number = 100*(Positive / (Positive + Negative))
        if (number != ''):
            Sentiment = ((100*(Negative / (Positive + Negative)))*(prc/mean_prc))*mcf

    new_row = {"Date":date, "Hour": Hour,"Negative":Negative, "Positive":Positive, "Sentiment":Sentiment, "Polarity": Polarity, "Subjectivity":Subjectivity, "Articles":sentence, "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "PressReleaseCount":row["PressReleaseCount"], "MarketCapFactor":MarketCapFactor}
    rows_list.append(new_row)

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "Negative", "Positive", "Sentiment", "Polarity","Articles","Title","Text", "Company", "PressReleaseCount", "MarketCapFactor"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourlyGeneralInquirer.pkl")

# 3) TRADING HOURS 09-16

In [ ]:
del df 

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_09_15.pkl', compression=None)

In [ ]:
df = pd.DataFrame()

df[['Negative']] = ''
df[['Positive']] = ''
df[['Sentiment']] = ''
df[['Adjectives']] = ''
df[['Verbs']] = ''
df[['Adverbs']] = ''
df[['Nouns']] = ''
df[["Articles"]] = ''
df[["Polarity"]] = ''
df[["Subjectivity"]] = ''

for date, row in df_articles.T.iteritems():
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'Articles'] = sentence
    prc = row['PressReleaseCount']

    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)
    df.at[date, 'Negative'] = score["Negative"]
    df.at[date, 'Positive'] = score["Positive"]
    df.at[date, 'Polarity'] = score["Polarity"]
    df.at[date, 'Subjectivity'] = score["Subjectivity"]
    if ((score['Positive'] + score['Negative']) != 0):
        number = 100*(score['Positive'] / (score['Positive'] + score['Negative']))
        if (number != ''):
            df.at[date, 'Sentiment'] = ((100*(score['Negative'] / (score['Positive'] + score['Negative'])))*(prc/mean_prc))

In [ ]:
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15)GeneralInquirer.pkl")

In [ ]:
del df

# 4) Not Trading Hours 16-09

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_16_08.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['Negative']] = ''
df[['Positive']] = ''
df[["Polarity"]] = ''
df[["Subjectivity"]] = ''
df[['Sentiment']] = ''
df[["Articles"]] = ''

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'Articles'] = sentence
    prc = row['PressReleaseCount']
    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)
    df.at[date, 'Negative'] = score['Negative']
    df.at[date, 'Positive'] = score['Positive']
    if ((score['Positive'] + score['Negative']) != 0):
        number = 100*(score['Positive'] / (score['Positive'] + score['Negative']))
        if (number != ''):
            df.at[date, 'Sentiment'] = ((100*(score['Negative'] / (score['Positive'] + score['Negative'])))*(prc/mean_prc))

In [ ]:
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08_GeneralInquirer.pkl")

In [ ]:
del df

# 5) TIME

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df_articles.loc[df_articles["Date"]>"2020-06-01"]

In [ ]:
rows_list = []
for date, row in df_articles.T.iteritems():

    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))

    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)
    Hour = row["Hour"]
    Time = row["Time"]
    Date = row["Date"]
    Negative = score['Negative']
    Positive = score['Positive']
    Sentiment=0
    if ((Positive + Negative) != 0):
        number = 100*(Positive / (Positive + Negative))
        if (number != ''):
            Sentiment = 100*((Negative) / (Positive + Negative))
    two_grams = generate_ngrams(sentence,2)
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    PolarityB = ""
    if (blob.sentiment.polarity > 0 ):
        PolarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        PolarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        PolarityB = "Neutral"

    new_row = {"Time":Time, "Date":Date, "Hour": Hour, "Sentiment":Sentiment, "Negative":Negative, "Positive":Positive, "Polarity": Polarity, "Subjectivity":Subjectivity,"PolarityB":PolarityB, "Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"]}
    rows_list.append(new_row)

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Time", "Date", "Hour", "Negative","Positive", "Sentiment", "Polarity", "Subjectivity", "PolarityB","Articles","Title","Text", "Company"])

In [ ]:
# print("Number of Negative words="+str(countNeg))
# print("Number of Positive words="+str(countPos))
# print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeGeneralInquirer.pkl")

In [ ]:
df.index = df.Date

In [ ]:
df

In [ ]:
df["Sentiment"].ewm(10).mean().plot(figsize=(20,10))

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTimeGeneralInquirer.pkl")

In [ ]:
df.index = df.Time

In [ ]:
df.loc[df["Company"]=="BBY"]["polarity"].plot(figsize=(20,10))

In [ ]:
df["sentiment"].ffill(inplace=True)